<a href="https://colab.research.google.com/github/jiangnanhugo/spring2026-tutorials/blob/main/chapter_recurrent-neural-networks/rnn-scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The following additional libraries are needed to run this
notebook. Note that running on Colab is experimental, please report a Github
issue if you have any problem.

# Recurrent Neural Network Implementation from Scratch
We are now ready to implement an RNN from scratch.
In particular, we will train this RNN to function
as a character-level language model
and train it on a corpus consisting of
the entire text of H. G. Wells' *The Time Machine*,
following the data processing steps.
We start by loading the dataset.


In [120]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F


## RNN Model

We begin by defining a class
to implement the RNN model.
Note that the number of hidden units `num_hiddens`
is a tunable hyperparameter.


In [140]:
class RNN(nn.Module):
    """The RNN model implemented from scratch."""
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.num_inputs = num_inputs
        self.num_hiddens = num_hiddens
        self.sigma = sigma

        # TODO: Initialize input-to-hidden weight matrix W_xh as an nn.Parameter
        # Shape: (num_inputs, num_hiddens)
        self.W_xh = nn.Parameter(torch.randn(num_inputs, num_hiddens)*sigma)

        # TODO: Initialize hidden-to-hidden (recurrent) weight matrix W_hh as an nn.Parameter
        # Shape: (num_hiddens, num_hiddens)
        self.W_hh = nn.Parameter(torch.randn(num_hiddens, num_hiddens) * sigma)

        # TODO: Initialize hidden bias b_h as an nn.Parameter
        # Shape: (num_hiddens,)
        self.b_h = nn.Parameter(torch.zeros(num_hiddens))

[**The `forward` method below defines how to compute
the output and hidden state at any time step,
given the current input and the state of the model
at the previous time step.**]
Note that the RNN model loops through
the outermost dimension of `inputs`,
updating the hidden state
one time step at a time.
The model here uses a $\tanh$ activation function.


In [141]:
def forward(self, inputs):
    # inputs: shape is (num_steps, batch_size, num_inputs).

    batch_size = inputs.shape[1]

    # TODO: Initialize the hidden state h_0 with zeros.
    # Shape should be: (batch_size, num_hiddens)
    state = torch.zeros(batch_size, self.num_hiddens, device=inputs.device)

    outputs = []
    for X in inputs:
        # X shape: (batch_size, num_inputs)
        # print("X shape", X.shape)
        # TODO: Compute next hidden state with the RNN recurrence:

        #   h_t = tanh( X @ W_xh + h_{t-1} @ W_hh + b_h )
        state = torch.tanh( X @ self.W_xh + state @ self.W_hh + self.b_h)


        outputs.append(state)

    # TODO: Return outputs and state
    return outputs, state

RNN.forward = forward

We can feed a minibatch of input sequences into an RNN model as follows.


In [142]:
batch_size, num_inputs, num_hiddens, seq_len = 2, 16, 32, 100
rnn = RNN(num_inputs, num_hiddens)
X = [torch.randn(batch_size, num_inputs) for _ in range(seq_len)]
outputs, state = rnn.forward(X)
print("outputs shape", outputs[0].shape, len(outputs))
print("state shape", state.shape)

AttributeError: 'list' object has no attribute 'shape'

## RNN-Based Language Model

The following `RNNLMScratch` class defines
an RNN-based language model,
where we pass in our RNN
via the `rnn` argument
of the `__init__` method.
When training language models,
the inputs and outputs are
from the same vocabulary.
Hence, they have the same dimension,
which is equal to the vocabulary size.
Note that we use perplexity to evaluate the model.
As discussed in :numref:`subsec_perplexity`, this ensures
that sequences of different length are comparable.


In [143]:
class RNNLM(nn.Module):
    """The RNN-based language model implemented from scratch."""
    def __init__(self, rnn, vocab_size:int, lr=0.01):
        super().__init__()

        ### save_hyperparameters
        self.rnn = rnn
        self.vocab_size = vocab_size
        self.lr = lr

        self.W_ho = nn.Parameter(torch.randn(self.rnn.num_hiddens, self.vocab_size) * self.rnn.sigma)
        self.b_o = nn.Parameter(torch.zeros(self.vocab_size))

        self.criterion = nn.CrossEntropyLoss()



### [**One-Hot Encoding**]

A one-hot encoding is a vector whose length
is given by the size of the vocabulary $N$,
where all entries are set to $0$,
except for the entry corresponding
to our token, which is set to $1$.
For example, if the vocabulary had five elements,
then the one-hot vectors corresponding
to indices 0 and 2 would be the following.


In [144]:
from torch.nn import functional as F
F.one_hot(torch.tensor([0, 2]), num_classes=100)

tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0]])

(**The minibatches that we sample at each iteration
will take the shape (batch size, number of time steps).
Once representing each input as a one-hot vector,
we can think of each minibatch as a three-dimensional tensor,
where the length along the third axis
is given by the vocabulary size (`len(vocab)`).**)
We often transpose the input so that we will obtain an output
of shape (number of time steps, batch size, vocabulary size).
This will allow us to loop more conveniently through the outermost dimension
for updating hidden states of a minibatch,
time step by time step
(e.g., in the above `forward` method).


In [145]:
def one_hot(self, X):
    # Input X: (batch_size, num_steps)
    # Output shape: (num_steps, batch_size, vocab_size)
    # TODO: convert each index into one-hot vector
    one_hot_seqs = F.one_hot(X.T, num_classes=self.vocab_size).float()
    return one_hot_seqs

RNNLM.one_hot=one_hot

In [146]:
def loss(self, y_hat, y):
        #   y_hat: (num_steps, batch_size, vocab) -> (num_steps*batch_size, vocab)
        #   y:     (num_steps, batch_size)       -> (num_steps*batch_size,)
        y_hat = y_hat.reshape(-1, y_hat.shape[-1])
        y = y.reshape(-1)
        return self.criterion(y_hat, y)
RNNLM.loss=loss

### Transforming RNN Outputs

The language model uses a fully connected output layer
to transform RNN outputs into token predictions at each time step.


In [159]:
def forward(self, X, state=None):
    # X: (batch_size, num_steps) token ids
    embs = self.one_hot(X)
    rnn_outputs, state = self.rnn(embs)
    output = self.output_layer(rnn_outputs)
    return output, state

RNNLM.forward = forward


In [160]:
def output_layer(self, rnn_outputs):
    outputs = []
    for ht in rnn_outputs:
      #  TODO: Apply the output projection to map hidden state -> vocab logits
      # output shape: (batch_size, vocab_size)
      outputs.append(torch.matmul(ht, self.W_ho) + self.b_o)

    return torch.stack(outputs, 1)

RNNLM.output_layer=output_layer

## [**Gradient Clipping**]



In [161]:
def grad_clipping_(model, theta):
    params = [p for p in model.parameters() if p.requires_grad and p.grad is not None]
    if not params:
        return
    norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
    if norm > theta:
        for p in params:
            p.grad[:] *= theta / (norm + 1e-12)

In [162]:
### the following data prep code is given


import re

def preprocess(text):
    # replaces every chunk of non-letters with a single space ' ',
    # A-Za-z = any English letter (uppercase or lowercase)
    # + = one or more in a row
    proc_text=re.sub('[^A-Za-z]+', ' ', text)
    return proc_text.lower()

def build_char_corpus(text: str):
    chars = list(preprocess(text))
    vocab = Vocab(chars)
    corpus = torch.tensor([vocab[c] for c in chars], dtype=torch.long)
    return corpus, vocab

def seq_data_iter_random(corpus, batch_size, num_steps, device):
    # TODO: Random sampling of subsequences.
    # corpus: (N,)
    N = len(corpus) - 1
    num_subseqs = (N - 1) // num_steps
    initial_indices = torch.randperm(num_subseqs, device=device) * num_steps

    def data(pos):
        return corpus[pos:pos + num_steps]

    for i in range(0, num_subseqs, batch_size):
        batch_indices = initial_indices[i:i + batch_size]
        X = torch.stack([data(int(j)) for j in batch_indices])
        Y = torch.stack([data(int(j) + 1) for j in batch_indices])
        yield X, Y

In [163]:
### the following dataset, vocabulary construction code is given

from pathlib import Path
import hashlib
import urllib.request
import re
from collections.abc import Iterable
class TimeMachine:
    """The Time Machine dataset."""
    def __init__(self, root="./data"):
        self.root = Path(root)

    def download(self):
        url = "https://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt"

        self.root.mkdir(parents=True, exist_ok=True)
        fname = self.root / "timemachine.txt"

        # Download only if missing
        if not fname.exists():
            urllib.request.urlretrieve(url, fname)

        with open(fname, "r", encoding="utf-8") as f:
            return f.read()


class Vocab:
    """Vocabulary for text."""
    def __init__(self, tokens=[], min_freq=0, reserved_tokens=[]):
        # Flatten a 2D list if needed

        if tokens and isinstance(tokens[0], list):
            tokens = [token for line in tokens for token in line]
        # Count token frequencies
        freq = {}
        for t in tokens:
          freq[t] = freq.get(t,0)+1


        # Sort by frequency (high -> low)
        self.token_freqs = sorted(freq.items(), key=lambda x: x[1], reverse=True)

        # The list of unique tokens
        self.idx_to_token=[]
        for token, freq in self.token_freqs:
          if freq >=min_freq:
            self.idx_to_token.append(token)
        self.idx_to_token.append(['<unk>'])

        self.idx_to_token = list(sorted(set(['<unk>'] + reserved_tokens + [
            token for token, freq in self.token_freqs if freq >= min_freq])))
        self.token_to_idx={}
        for idx, token in enumerate(self.idx_to_token):
          self.token_to_idx[token]=idx
        self.token_to_idx = {token: idx
                             for idx, token in enumerate(self.idx_to_token)}

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
      # Case 1: `tokens` is a single token (e.g., a string like "the")
      # Return its index if it exists; otherwise return the unknown-token index `self.unk`.
      if not isinstance(tokens, (list, tuple)):
          ### OOV out-of-vocabulary
          return self.token_to_idx.get(tokens, self.unk)

      # Case 2: `tokens` is a list/tuple of tokens (e.g., ["the", "time", "machine"])
      # Convert each token to its index, preserving order.
      # Uses self.__getitem__ recursively so the same "single token" logic applies.
      return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
      # If `indices` looks like a sequence (list/tuple/numpy array/torch tensor),
      # convert each element to an int index and map it to its token.
      # (Exclude strings/bytes so we don't iterate over characters.)
      if hasattr(indices, "__len__") and not isinstance(indices, (str, bytes)):
        temp=[]
        for i in indices:
          one_token=self.idx_to_token[int(i)]
          temp.append(one_token)
        return temp

        # return [self.idx_to_token[int(i)] for i in indices]

      # Otherwise, treat `indices` as a single scalar index and return one token.
      return self.idx_to_token[int(indices)]

    @property
    def unk(self):  # Index for the unknown token
        return self.token_to_idx['<unk>']

    def build(self, raw_text, vocab=None):
        # 1) Clean/normalize the raw text (e.g., lowercase, remove punctuation)
        #    then split it into a list of tokens (words/characters depending on tokenizer).

        ## tokens = self._tokenize(self._preprocess(raw_text))
        tokens = [sent.split(" ") for sent in self._preprocess(raw_text).split("\n")]

        vocab=Vocab(tokens)
        corpus = [vocab[token] for token in tokens]

        # 4) Return both the numeric corpus and the vocabulary used to build it.
        return corpus, vocab

    def _preprocess(self, text):
        # replaces every chunk of non-letters with a single space ' ',
        # A-Za-z = any English letter (uppercase or lowercase)
        # + = one or more in a row
        proc_text=re.sub('[^A-Za-z]+', ' ', text)
        return proc_text.lower()

tm = TimeMachine()
raw_text = tm.download()
print(raw_text[:10])
corpus, vocab = build_char_corpus(raw_text)
print(corpus[:10])


The Time M
tensor([21,  9,  6,  0, 21, 10, 14,  6,  0, 14])


In [164]:
# print(corpus, len(corpus))
# print(vocab.idx_to_token)
print(vocab.token_to_idx)

{' ': 0, '<unk>': 1, 'a': 2, 'b': 3, 'c': 4, 'd': 5, 'e': 6, 'f': 7, 'g': 8, 'h': 9, 'i': 10, 'j': 11, 'k': 12, 'l': 13, 'm': 14, 'n': 15, 'o': 16, 'p': 17, 'q': 18, 'r': 19, 's': 20, 't': 21, 'u': 22, 'v': 23, 'w': 24, 'x': 25, 'y': 26, 'z': 27}


In [165]:
def train_language_model(
    model,
    corpus,
    batch_size=2, num_steps=10, lr=0.001,
    max_epochs=10, grad_clip_val=1.0,
    device=None):

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    corpus = corpus.to(device)
    # TODO Pick a gradient optimzier

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss, total_tokens = 0.0, 0

        for X, Y in seq_data_iter_random(corpus, batch_size, num_steps, device):
            # TODO:
            # 1. compute output for the input X
            # 2. compute the loss
            # 3. compute the gradient
            # 4. update the params with gradient.
            output,_ = model.forward(X)
            loss = model.loss(output, Y)
            loss.backward()
            grad_clipping_(model, grad_clip_val)
            optimizer.step()

            # optimizer.zero_grad()
            # logits, _ = model(X)
            # l = model.loss(logits, Y)
            # l.backward()
            # grad_clipping_(model, grad_clip)
            # optimizer.step()
            print("loss is", loss.item())
            total_loss += loss.item() * Y.numel()
            total_tokens += Y.numel()

        ppl = math.exp(total_loss / total_tokens)
        print(f"epoch {epoch:03d} | ppl {ppl:.3f}")

## Training

Using *The Time Machine* dataset (`data`),
we train a character-level language model (`model`)
based on the RNN (`rnn`) implemented from scratch.
Note that we first calculate the gradients,
then clip them, and finally
update the model parameters
using the clipped gradients.


In [166]:
# --- build model ---
rnn = RNN(num_inputs=len(vocab), num_hiddens=32)
print("vocab size is", len(vocab))
model = RNNLM(rnn=rnn, vocab_size=len(vocab), lr=1.0)

# --- train (no d2l) ---
train_language_model(
    model, corpus,
    batch_size=10,
    num_steps=32,
    lr=1.0,
    max_epochs=2,
    grad_clip_val=1.0
)

vocab size is 28
loss is 3.332266330718994
loss is 10.07061767578125
loss is 17.546955108642578
loss is 18.324111938476562
loss is 21.8951473236084
loss is 29.453746795654297
loss is 33.162540435791016
loss is 38.87531280517578
loss is 45.82014465332031
loss is 50.79086685180664
loss is 59.223976135253906
loss is 56.16460418701172
loss is 59.54011154174805
loss is 65.16771697998047
loss is 72.43693542480469
loss is 75.22645568847656
loss is 75.84001159667969
loss is 80.67759704589844
loss is 75.68684387207031
loss is 76.50135803222656
loss is 73.53205108642578
loss is 68.91926574707031
loss is 72.29574584960938
loss is 61.482643127441406
loss is 59.724754333496094
loss is 58.97252655029297
loss is 52.11430740356445
loss is 48.05311965942383
loss is 39.18657684326172
loss is 38.88835906982422
loss is 41.927547454833984
loss is 34.623443603515625
loss is 36.08870315551758
loss is 37.68882369995117
loss is 39.72760772705078
loss is 36.55720138549805
loss is 34.7521858215332
loss is 33.586

KeyboardInterrupt: 

## Decoding

Once a language model has been learned,
we can use it not only to predict the next token
but to continue predicting each subsequent one,
treating the previously predicted token as though
it were the next in the input.
Sometimes we will just want to generate text
as though we were starting at the beginning
of a document.
However, it is often useful to condition
the language model on a user-supplied prefix.
For example, if we were developing an
autocomplete feature for a search engine
or to assist users in writing emails,
we would want to feed in what they
had written so far (the prefix),
and then generate a likely continuation.


[**The following `predict` method
generates a continuation, one character at a time,
after ingesting a user-provided `prefix`**].
When looping through the characters in `prefix`,
we keep passing the hidden state
to the next time step
but do not generate any output.
This is called the *warm-up* period.
After ingesting the prefix, we are now
ready to begin emitting the subsequent characters,
each of which will be fed back into the model
as the input at the next time step.


In [167]:
def predict(self, prefix, num_preds, vocab, device=None):
    device = device or next(self.parameters()).device
    self.eval()

    # Start with the full prefix (so the returned string begins with prefix)
    outputs = [vocab[c] for c in prefix]
    state = None

    with torch.no_grad():
        # Warm-up: feed prefix except the last char to build state
        for c in prefix[:-1]:
            X = torch.tensor([[vocab[c]]], dtype=torch.long, device=device)  # (1, 1)
            _, state = self(X, state)

        # Now generate num_preds new chars, starting from the last prefix char
        last_id = outputs[-1]
        for _ in range(num_preds):
            X = torch.tensor([[last_id]], dtype=torch.long, device=device)  # (1, 1)
            logits, state = self(X, state)  # logits: (1, 1, vocab_size)
            last_id = int(logits[0, 0].argmax(dim=-1).item())
            outputs.append(last_id)

    return ''.join(vocab.idx_to_token[i] for i in outputs)

RNNLM.predict = predict

In the following, we specify the prefix
and have it generate 20 additional characters.


In [168]:
model.predict('it has', 20, vocab)

'it hasasasasasasasasasasas'

While implementing the above RNN model from scratch is instructive, it is not convenient.
In the next section, we will see how to leverage deep learning frameworks to whip up RNNs
using standard architectures, and to reap performance gains
by relying on highly optimized library functions.


## Summary

We can train RNN-based language models to generate text following the user-provided text prefix.
A simple RNN language model consists of input encoding, RNN modeling, and output generation.
During training, gradient clipping can mitigate the problem of exploding gradients but does not address the problem of vanishing gradients. In the experiment, we implemented a simple RNN language model and trained it with gradient clipping on sequences of text, tokenized at the character level. By conditioning on a prefix, we can use a language model to generate likely continuations, which proves useful in many applications, e.g., autocomplete features.



